In [1]:
import os.path
import glob
import datetime
import pandas as pd

In [2]:
"""
This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program. If not, see <https://www.gnu.org/licenses/>.

Equivalent R code found at: 
                    https://github.com/ahasverus/elbow
                    R code author : Nicolas Casajus (2020)

This python version authored by: Chathura Jayalath (2023)
"""
import pandas as pd

from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

def get_elbow(in_xvals, in_yvals, in_plot=False, in_image_path="./elbow.png"):
    tdf = pd.DataFrame({"x": in_xvals, "y": in_yvals})
    model = LinearRegression().fit(tdf["x"].values.reshape((-1, 1)), tdf["y"].values)
    tdf["constant"] = model.intercept_ + model.coef_ * tdf["x"]
    pos = tdf.shape[0] // 2 - 1
    if tdf.loc[pos, "constant"] < tdf.loc[pos][1]:
        print("min")
        ymin = min(tdf.iloc[:, 1])
        tdf["benefits"] = ymin + (tdf.iloc[:, 1] - tdf["constant"])
        maxi = tdf.loc[tdf["benefits"].idxmax()]
    else:
        print("max")
        ymax = max(tdf.iloc[:, 1])
        tdf["benefits"] = ymax - (tdf["constant"] - tdf.iloc[:, 1])
        maxi = tdf.loc[tdf["benefits"].idxmin()]
    if in_plot:
        fig1, ax1 = plt.subplots()
        ax1.plot(tdf["x"].values, tdf["y"].values, label="data")
        ax1.plot(tdf["x"].values, tdf["benefits"].values, label="benefits")
        ax1.scatter(x=maxi["x"], y= maxi["y"], c='r') #label='Elbow Point')
        ax1.scatter(x=maxi["x"], y= maxi["benefits"], c='r')#, label='Elbow Est.')
        ax1.axvline(maxi["x"], color='red', linewidth=.5)
        ax1.text(maxi["x"], maxi["y"], "{:.2f} , {:.2f}".format(maxi["x"],maxi["y"]), fontsize = 20) 
        plt.legend()
        plt.savefig(in_image_path)
        # plt.show()
    print(f"idx: {maxi.name}")
    return maxi

In [3]:
class CalcWY:

    def __init__(self, 
                 data_dir="../../data/april1/*.csv*", 
                 start_date_w = datetime.datetime(2023, 5, 15),
                 event_date = datetime.datetime(2023, 10, 7),
                 sentiment_measure="Consistancy", # Mode or Consistancy
                 frequency="1D"):
        self.data_dir = data_dir
        self.start_date_w = start_date_w
        self.event_date = event_date
        self.sentiment_measure = sentiment_measure
        self.sentiment_measure_col = f"Sentiment{self.sentiment_measure}"
        self.frequency = frequency
        # -- Dynamics --
        self.y_delta_length_days = 14
        self.post_count_threshold = 0
        self.df_all = None # all data for W
        self.df_wc = None # working copy of W
        self.end_date_w = None
        self.end_date_y2 = None
        self.author_brand = None
        self.author_to_brand = None
        self.sentiment_matrix = None
        self.sentiment_table = None
    
    def read_brandwatch_data(self):
        """ Read data """
        df_all = pd.concat([ pd.read_csv(f, 
                                          skiprows=6, 
                                          parse_dates=["Date"], 
                                          usecols=["Author","Sentiment","Date","Url","Title"]) 
                             for f in glob.glob(self.data_dir)]).sort_values("Date")
        df_all.dropna(subset=["Author","Date","Sentiment"], inplace=True)
        # comment the following if not only Twitter data is used!
        df_all.drop_duplicates("Url", inplace=True) # remove duplicate tweets by tweetid
        # wdf.drop(columns=["Url"], inplace=True)
        # convert all tweet text to lower case
        df_all["Title"] = df_all["Title"].apply(lambda x: x.lower() if type(x) is str else x)
        print("DataFrame Shape :", df_all.shape)
        print("Unique User Count :", df_all["Author"].nunique())
        # drop empty text tweets
        df_all.drop(df_all[df_all["Title"].isna()].index, inplace=True)
        print("Dropped Empty Titles")
        print("DataFrame Shape :", df_all.shape)
        print("Unique User Count :", df_all["Author"].nunique())
        df_all.reset_index(drop=True, inplace=True)
        self.df_all = df_all
        self.df_wc = self.df_all
        return self.df_all

    def setup_dates(self, y_delta_length_days = 14):
        self.y_delta_length_days = y_delta_length_days
        
        """
            Data is divided as follows:
            [ ... W ... ][ ... Y1 ...][ ... Y2 ... ]
            tA          tB           tC            tD
        tA = start_date_w
        tB = end_date_w = start_date_y1
        tC = event_date = end_date_y2 = start_date_y1
        tD = end_date_y2
        tC - tB = tD - tC = y_delta_length_days
        """
        
        self.end_date_w = self.event_date - datetime.timedelta(days=self.y_delta_length_days)
        self.end_date_y2 = self.event_date + datetime.timedelta(days=self.y_delta_length_days)
        
        print("W length :", self.end_date_w - self.start_date_w)
        print("Y1 length == Y2 length :", self.y_delta_length_days)
        print("tA =", self.start_date_w)
        print(f"   W:{self.end_date_w - self.start_date_w} ")
        print("tB =", self.end_date_w)
        print(f"   Y1:{self.y_delta_length_days} ")
        print("tC =", self.event_date)
        print(f"   Y2:{self.y_delta_length_days} ")
        print("tD =", self.end_date_y2)

        print("\nUsers in W =", self.df_all[
              (self.df_all["Date"] >= self.start_date_w) & 
              (self.df_all["Date"] < self.end_date_w)
              ]["Author"].nunique() )
        
        print("\nUsers in Y1 =", self.df_all[
              (self.df_all["Date"] >= self.end_date_w) & 
              (self.df_all["Date"] < self.event_date)
              ]["Author"].nunique() )
        
        print("\nUsers in Y2 =", self.df_all[
              (self.df_all["Date"] >= self.event_date) & 
              (self.df_all["Date"] < self.end_date_y2)
              ]["Author"].nunique() )

    @staticmethod
    def get_user_posts(df):
        """ returns number of users with the given number of posts """
        return df.groupby("Author").size().value_counts()

    @staticmethod
    def get_size(df):
        """ saves number of posts each author have done """
        wdf_size=df.groupby("Author").size().sort_values(ascending=False).reset_index(name='Count')
        wdf_size["Fraction"] = wdf_size["Count"] / df.shape[0]
        return wdf_size

    @staticmethod
    def elbow_value(wdf_size):
        elbow_data = wdf_size["Count"].value_counts().rename("UserCount").rename_axis("NumTweets").reset_index()
        return get_elbow(elbow_data["NumTweets"], elbow_data["UserCount"], in_plot=True, in_image_path="./elbow.png")

    def set_brand_cols(self, brand_column_key = {"McDonalds":"mcdonald","Starbucks":"starbuck", "Nike":"nike"}):
        self.brand_column_key = brand_column_key
        self.brand_columns = []
        for col in self.brand_column_key:
            self.brand_columns.append(col)
            self.df_all[col] = self.df_all["Title"].apply(lambda x: self.brand_column_key[col] in x)

        self.df_all.drop(index = self.df_all[~self.df_all[self.brand_columns].any(axis=1)].index, inplace=True)
        self.df_all.reset_index(drop=True, inplace=True)
        return self.df_all

    def calculate_author_brand(self):
        author_columns = []
        for col in self.brand_columns:
            author_col = self.df_all[["Author", col]].groupby("Author").apply(lambda gwdf: gwdf[col].sum()).rename(f"{col}_Count").reset_index()
            author_columns.append(author_col)

        author_brand = author_columns[0]
        for ac in author_columns[1:]:
            author_brand = author_brand.merge(ac, how="outer", on="Author", validate="one_to_one")
        author_brand.set_index("Author", inplace=True)
        author_brand["Counts_Sum"] = author_brand.sum(axis=1)
        author_brand["Max_Type"] = author_brand[[f"{col}_Count" for col in self.brand_columns]].idxmax(axis=1).apply(lambda x: x[:-len("_Count")])
        self.author_brand = author_brand
        return self.author_brand

    # def get_all_W_data(self):
    #     return self.df_all[(self.df_all["Date"] >= self.start_date_w) & (self.df_all["Date"] < self.end_date_w)]

    def get_wc_W_data(self):
        return self.df_wc[(self.df_wc["Date"] >= self.start_date_w) & (self.df_wc["Date"] < self.end_date_w)]

    def add_sentiment(self):
        sentiment_to_number = {"none": 1, "positive": 2, "neutral": 3, "negative": 4}
        self.df_all["SentimentNo"] = self.df_all["Sentiment"].apply(lambda x: sentiment_to_number[x])

    def apply_author_brand_constraint(self):
        self.author_to_brand = self.author_brand['Max_Type'].to_dict()
        self.df_all["AuthorBrand"] = self.df_all["Author"].apply(lambda x: self.author_to_brand[x])
        # remove tweets in which Authors talk about other brands than the one they mostly talk about
        self.df_all.drop(self.df_all[self.df_all[self.brand_columns + ['AuthorBrand']].apply(lambda row: not row[row["AuthorBrand"]], axis=1)].index, inplace=True)

    def filter_authors_wc(self, post_count_threshold):
        self.post_count_threshold = post_count_threshold
        self.author_brand_wc = self.author_brand[self.author_brand["Counts_Sum"] > self.post_count_threshold]
        self.df_wc = self.df_all[self.df_all["Author"].isin(self.author_brand_wc.index)]
        return self.author_brand_wc

    @staticmethod
    def calculate_sentiment_table(df, sentiment_measure, sentiment_measure_col, frequency):
        if "Mode" == sentiment_measure:
            def get_avg_sentiment(df):
                # print(df.head(), df["SentimentNo"].mode().sort_values().iloc[0])
                return df["SentimentNo"].mode().sort_values().iloc[0]

            pvt = df[["Date", "Author", "SentimentNo"]].set_index("Date").groupby([pd.Grouper(freq=frequency), 'Author']).apply(lambda tdf: get_avg_sentiment(tdf)).rename(sentiment_measure_col).reset_index()
        elif "Consistancy" == sentiment_measure:
            def get_sentiment_consistancy(df):
                # 0 if no data, 1 if mixed, 2 if consistant
                num_sentiments = df["SentimentNo"].nunique()
                if df.shape[0] == 0:
                    return 0
                if num_sentiments > 1:
                    return 1
                elif num_sentiments == 1:
                    return 2
                else:
                    return 0
            
            pvt = df[["Date", "Author", "SentimentNo"]].set_index("Date").groupby([pd.Grouper(freq=frequency), 'Author']).apply(lambda tdf: get_sentiment_consistancy(tdf)).rename(sentiment_measure_col).reset_index()
        else:
            raise ValueError('Sentiment Measure should be one of {"Mode", "Consistancy"}. Given values is {}'.format(sentiment_measure))

        return pvt

    def calculate_W_matrix(self):
        wdata = self.get_wc_W_data()
        self.W_sentiment_table = CalcWY.calculate_sentiment_table(wdata, self.sentiment_measure, self.sentiment_measure_col, self.frequency)
        output_wdf = self.W_sentiment_table.pivot_table(index="Author", columns="Date", values=self.sentiment_measure_col, fill_value=0)
        output_wdf.to_csv("./Outputs/W_data_f{}_t{}_y{}_{}.csv".format(self.frequency, self.post_count_threshold, self.y_delta_length_days, self.sentiment_measure))
        self.sentiment_matrix = output_wdf
        return self.sentiment_matrix
        
    def get_wc_Y1_data(self):
        return self.df_wc[(self.df_wc["Date"] >= self.end_date_w) & (self.df_wc["Date"] < self.event_date)]

    def get_wc_Y2_data(self):
        return self.df_wc[(self.df_wc["Date"] >= self.event_date) & (self.df_wc["Date"] < self.end_date_y2)]

    def calculate_Y_table(self):
        y1_data = self.get_wc_Y1_data()
        y2_data = self.get_wc_Y2_data()

        if "Mode" == self.sentiment_measure:        
            y1_author_sentiment_table = y1_data[["Author", "Sentiment"]].groupby("Author").value_counts().rename("SentimentCount").reset_index().pivot(index="Author", columns="Sentiment", values="SentimentCount").fillna(0)
            y2_author_sentiment_table = y2_data[["Author", "Sentiment"]].groupby("Author").value_counts().rename("SentimentCount").reset_index().pivot(index="Author", columns="Sentiment", values="SentimentCount").fillna(0)
            author_before_after = y1_author_sentiment_table.join(y2_author_sentiment_table, how="outer", lsuffix="_before", rsuffix="_after")
            author_before_after["diff_before"] = author_before_after["positive_before"] - author_before_after["negative_before"]
            author_before_after["diff_after"] = author_before_after["positive_after"] - author_before_after["negative_after"]
            
        elif "Consistancy" == self.sentiment_measure:
            self.y1_sentiment_table = CalcWY.calculate_sentiment_table(y1_data, self.sentiment_measure, self.sentiment_measure_col, self.frequency)
            self.y1_matrix = self.y1_sentiment_table.groupby("Author").value_counts().rename("Count").reset_index().pivot_table(index="Author", columns="SentimentConsistancy", values="Count", fill_value=0)
            
            self.y2_sentiment_table = CalcWY.calculate_sentiment_table(y2_data, self.sentiment_measure, self.sentiment_measure_col, self.frequency)
            self.y2_matrix = self.y2_sentiment_table.groupby("Author").value_counts().rename("Count").reset_index().pivot_table(index="Author", columns="SentimentConsistancy", values="Count", fill_value=0)
            
            author_before_after = self.y1_matrix.join(self.y2_matrix, how="outer", lsuffix="_before", rsuffix="_after")
            author_before_after["diff_before"] = author_before_after["1_before"] - author_before_after["2_before"]
            author_before_after["diff_after"] = author_before_after["1_after"] - author_before_after["2_after"]
            
        else:
            raise ValueError('Sentiment Measure should be one of {"Mode", "Consistancy"}. Given values is {}'.format(self.sentiment_measure))
        
        author_before_after["Y"] = author_before_after[["diff_before", "diff_after"]].apply(lambda row: 1 if abs(abs(row["diff_after"]) - abs(row["diff_before"])) > 1 else 0, axis=1)

        print( author_before_after["Y"].value_counts() )
        print( author_before_after["Y"].value_counts() / author_before_after.shape[0] )

        author_before_after.to_csv("./Outputs/Y_data_y{}_{}.csv".format(self.y_delta_length_days, self.sentiment_measure))
        
        self.author_before_after = author_before_after
        return author_before_after


In [4]:
def process(measure_type, ylength, post_threshold):
    print("--------", measure_type, ylength, post_threshold, "--------\n")
    cwy = CalcWY(data_dir="../../data/april1/*.csv*", 
                 start_date_w = datetime.datetime(2023, 5, 15),
                 event_date = datetime.datetime(2023, 10, 7),
                 sentiment_measure=measure_type, # Mode or Consistancy
                 frequency="1D")
    cwy.read_brandwatch_data()
    cwy.setup_dates(ylength)
    cwy.set_brand_cols()
    ab = cwy.calculate_author_brand()
    cwy.add_sentiment()
    cwy.apply_author_brand_constraint()
    cwy.filter_authors_wc(post_threshold)
    cwy.calculate_W_matrix()
    cwy.calculate_Y_table()
    print("-------- DONE --------\n")

In [5]:
for measure_type in ["Consistancy", "Mode"]:
    for ylength in [14, 30]:
        process(measure_type, ylength, 20)

-------- Consistancy 14 20 --------

DataFrame Shape : (345521, 5)
Unique User Count : 922
Dropped Empty Titles
DataFrame Shape : (345521, 5)
Unique User Count : 922
W length : 131 days, 0:00:00
Y1 length == Y2 length : 14
tA = 2023-05-15 00:00:00
   W:131 days, 0:00:00 
tB = 2023-09-23 00:00:00
   Y1:14 
tC = 2023-10-07 00:00:00
   Y2:14 
tD = 2023-10-21 00:00:00

Users in W = 889

Users in Y1 = 387

Users in Y2 = 402
Y
0    293
Name: count, dtype: int64
Y
0    1.0
Name: count, dtype: float64
-------- DONE --------

-------- Consistancy 30 20 --------

DataFrame Shape : (345521, 5)
Unique User Count : 922
Dropped Empty Titles
DataFrame Shape : (345521, 5)
Unique User Count : 922
W length : 115 days, 0:00:00
Y1 length == Y2 length : 30
tA = 2023-05-15 00:00:00
   W:115 days, 0:00:00 
tB = 2023-09-07 00:00:00
   Y1:30 
tC = 2023-10-07 00:00:00
   Y2:30 
tD = 2023-11-06 00:00:00

Users in W = 881

Users in Y1 = 503

Users in Y2 = 501
Y
0    317
Name: count, dtype: int64
Y
0    1.0
Name: 